In [1]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
import math

In [2]:
def make_radar_angles(n_axes: int) -> np.ndarray:
    """Angles for radar axes, closing the loop."""
    angles = np.linspace(0, 2 * math.pi, n_axes, endpoint=False)
    return np.concatenate([angles, angles[:1]])

def profile_values(
    skills: dict[str, float],
) -> tuple[list[str], list[float]]:
    """Return chart labels and values in dictionary order."""
    categories = list(skills)
    values = [max(1.0, min(3.0, float(value))) for value in skills.values()]
    return categories, values


def plot_radar(
    values: list[float],
    categories: list[str],
    color: str,
    alpha: float,
    ax: plt.Axes| None = None,
):
    """Plot a radar profile on given axis (create one if none)."""
    vals = values + values[:1]  # close polygon

    n = len(categories)
    angles = make_radar_angles(n)

    if ax is None:
        warnings.warn("No Axis object found, bulding it from scratch")
        fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True), dpi=180)

    # set up everything here: angles, ticks, names, ...
    if n % 2 == 1:
        # odd number of verteces: straight at bottom
        offset_step = 0
    elif n > 2:
        # even number: symmetric, rotate half a step from bottom
        offset_step = math.pi / n
    ax.set_theta_offset(-math.pi / 2 + offset_step)

    ax.set_theta_direction(-1)
    ax.set_ylim(0, 3)

    # Radial ticks replaced with novice/intermediate/expert (invisible)
    ax.set_yticks([0.0, 1.0, 2.0, 3.0])
    ax.set_yticklabels([])

    # Category names around the circle, first delete then set them manually
    ax.set_xticks([])

    label_radius = 1.1 * ax.get_ylim()[1]    # distance > 1.1 current y max

    # extra padding since more space is needed for angles ~90 and angles ~ 270

    for angle, category in zip(angles[:-1], categories):

        extra_pad = 0.5*np.abs(math.sin(angle - offset_step))

        ax.text(
            angle,
            label_radius + extra_pad,
            category,
            ha="center",
            va="center",
            fontsize=10,
        )
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([])

    # Plot
    ax.plot(angles, vals, color=color, linewidth=2)
    ax.fill(angles, vals, color=color, alpha=alpha)

    return ax


In [ ]:
current_skills = {
    "Research Framing \n& Experimental Design": 2.3,
    "Statistical Modeling \n& Inference": 2.5,
    "Machine Learning \n& Evaluation": 2.3,
    "Explainability, Robustness \n& Responsible AI": 3.0,
    "Data Analysis \n& Visualization": 2.8,
    "Scientific Software \n& Reproducibility": 2.0,
    "Research-to-Product \nTranslation": 1.8,
    "Technical Communication \n& Collaboration": 2.0,
}

categories, current_values = profile_values(current_skills)

# Create a single-profile capability map
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True), dpi=100)
ax.set_title("My R&D Data Science Focus", va="bottom", fontsize=14, pad=45)

plot_radar(current_values, categories, color="navy", alpha=0.20, ax=ax)
plt.savefig("skills-chart-profile.png", bbox_inches="tight")
plt.show()